In [ ]:
import pandas as pd
import os

# =============================================================
#               CONFIGURATION - CHANGE ONLY HERE IF NEEDED
# =============================================================

DAYS_IN_MONTH = 26   # ← change this (e.g. 22, 25, 26, 28...)

MAIN_FILE    = r"D:/Tushar/main_with_subs_only.xlsx"
INDENT_FILE  = r"D:/PPC Plan/Monthly Indent/Monthly Indent.xlsx"

# Column names — updated according to your explanation
COL_CHILD          = "Main_Label"       # ← child part
COL_SWITCH_BOM     = "Sub_Label"        # ← switch / parent part
COL_QTY_PER_SWITCH = "Sub_Count"        # ← how many child per switch

# In Monthly Indent file — please confirm / adjust these two:
COL_SWITCH_INDENT  = "Switch Part"      # ← most likely column name — change if different
COL_MONTHLY_QTY    = "Monthly Indent"   # ← most likely — change if different

# Output file (saved in the folder where you run this script)
OUTPUT_FILENAME = "2_Day_Qty_Child_Parts.xlsx"

# =============================================================
#                     MAIN LOGIC
# =============================================================

print("Reading files...")
try:
    df_bom   = pd.read_excel(MAIN_FILE)
    df_plan  = pd.read_excel(INDENT_FILE)
except Exception as e:
    print("\nERROR reading files:\n", e)
    input("\nPress Enter to exit...")
    exit()

df_bom.columns  = df_bom.columns.str.strip()
df_plan.columns = df_plan.columns.str.strip()

print(f"Rows in BOM file:   {len(df_bom):,}")
print(f"Rows in plan file:  {len(df_plan):,}\n")

# Merge monthly plan qty into the BOM table
print("Matching switches and calculating...")
merged = df_bom.merge(
    df_plan[[COL_SWITCH_INDENT, COL_MONTHLY_QTY]],
    left_on  = COL_SWITCH_BOM,
    right_on = COL_SWITCH_INDENT,
    how      = "left"
)

# Calculate total monthly requirement for each child
merged["Monthly_Req_Child"] = merged[COL_MONTHLY_QTY].fillna(0) * merged[COL_QTY_PER_SWITCH].fillna(0)

# Sum per child part
result = merged.groupby(COL_CHILD, as_index=False)["Monthly_Req_Child"].sum()
result = result.rename(columns={"Monthly_Req_Child": "Total_Monthly_Req"})

# Daily + 2-day
result["Daily_Req"]   = (result["Total_Monthly_Req"] / DAYS_IN_MONTH).round(2)
result["2_Day_Qty"]   = (result["Daily_Req"] * 2).round(2)
result["Total_Monthly_Req"] = result["Total_Monthly_Req"].round(1)

# Final table
final = result[[COL_CHILD, "Total_Monthly_Req", "Daily_Req", "2_Day_Qty"]].sort_values("2_Day_Qty", ascending=False)

# Save
output_path = os.path.join(os.getcwd(), OUTPUT_FILENAME)
try:
    final.to_excel(output_path, index=False)
    print("\n" + "═" * 70)
    print("                  SUCCESS")
    print("═" * 70)
    print(f"Saved to:       {output_path}")
    print(f"Full path:      {os.path.abspath(output_path)}")
    print(f"Child parts:    {len(final):,}")
    print(f"Days used:      {DAYS_IN_MONTH}")
    print("═" * 70 + "\n")
except Exception as e:
    print("Error saving file:\n", e)

input("Press Enter to close...")